# Intelligent Document Processing (IDP) with Blueprints in Bedorck Data Automation

Workflow:
-
- Upload scanned US bank check to an S3 bucket
- Select custom Blueprint based on use-case
- Select BDA project (or create one)
- API InvokeDataAutomationAsync is invoked for IDP
- View custom output with Blueprint

## Setup some S3 parameters and configuration that will be used throughout this notebook.

In [1]:
%pip install --no-warn-conflicts "boto3>=1.37.6" itables==2.2.4 PyPDF2==3.0.1 --upgrade -qq
%load_ext autoreload
%autoreload 1

Note: you may need to restart the kernel to use updated packages.


Before we get to the part where we invoke BDA with our sample artifacts, let's setup some parameters and configuration that will be used throughout this notebook.

In [2]:
import boto3
import json
from IPython.display import JSON, IFrame
import sagemaker
import pandas as pd
from utils import display_functions, helper_functions
from pathlib import Path
import os

session = sagemaker.Session()
default_bucket = session.default_bucket()
current_region = boto3.session.Session().region_name

sts_client = boto3.client('sts')
account_id = sts_client.get_caller_identity()['Account']

# Initialize Bedrock Data Automation client
bda_client = boto3.client('bedrock-data-automation')
bda_runtime_client = boto3.client('bedrock-data-automation-runtime')
s3_client = boto3.client('s3')

bda_s3_input_location = f's3://{default_bucket}/bda/input'
bda_s3_output_location = f's3://{default_bucket}/bda/output'

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/sagemaker-user/.config/sagemaker/config.yaml


## Prepare scanned bank check

The scanned bank check document is uploaded to S3 bucket "input" location.

In [3]:
local_download_path = 'data/documents'
local_file_name = 'Demo-check5-handwritten.PNG'
local_file_path = os.path.join(local_download_path, local_file_name)
#(bucket, key) = utils.get_bucket_and_key(document_url)
#response = s3_client.download_file(bucket, key, local_file_path)

document_s3_uri = f'{bda_s3_input_location}/{local_file_name}'

target_s3_bucket, target_s3_key =  helper_functions.get_bucket_and_key(document_s3_uri)
s3_client.upload_file(local_file_path, target_s3_bucket, target_s3_key)

print(f"Downloaded file to: {local_file_path}")
print(f"Uploaded file to S3: {target_s3_key}")
print(f"document_s3_uri: {document_s3_uri}")

Downloaded file to: data/documents/Demo-check5-handwritten.PNG
Uploaded file to S3: bda/input/Demo-check5-handwritten.PNG
document_s3_uri: s3://sagemaker-us-east-1-578787088678/bda/input/Demo-check5-handwritten.PNG


### View Sample Document

In [4]:
IFrame(local_file_path, width=600, height=400)

## Create custom blueprints and project

From the "bedrock-data-automation-public-us-bank-check" catalogue blueprint, we customized fields to retrieve additional details from the document. Using the "schema_path" for the customized blueprint, either a new blueprint is created or updated in exuists with anticipated Stage and Schema. In this use-case we use a blueprint that displays if a "Memo" is present and the value for it along with other fields like Acccount Bumber, Routing Number and Check Number. 

We use the `create_blueprint` operation (or `update_blueprint` to update an existing blueprint) in the  `boto3` API to create/update the blueprint. Each blueprint that you create is an AWS resource with its own blueprint ID and ARN. 

In [5]:
# create blueprint using Boto3
blueprints = [
    {
        "name": 'Check-blueprint',
        "description": 'Blueprint for US Bank Check',
        "type": 'DOCUMENT',
        "stage": 'LIVE',
        "schema_path": 'data/blueprints/Check-blueprint.json'
    }
]


In [6]:
blueprint_arns = []
for blueprint in blueprints:
    with open(blueprint['schema_path']) as f:
        blueprint_schema = json.load(f)
        blueprint_arn = helper_functions.create_or_update_blueprint(
            bda_client, 
            blueprint['name'], 
            blueprint['description'], 
            blueprint['type'],
            blueprint['stage'],
            blueprint_schema
        )
        blueprint_arns += [blueprint_arn]
print(blueprint_arns)

Found existing blueprint with name=Check-blueprint, updating Stage and Schema
['arn:aws:bedrock:us-east-1:578787088678:blueprint/60638c55f4fb']


The `update_data_automation_project` API takes a project name, description, stage (LIVE / DEVELOPMENT), the standard output configuration and a custom output configuration as input. We are only focussing on the custom output in this notebook, so we leave the standard output configuration as empty so BDA will use the defaults. Additionally, we use a custom configuration with the arn for the recommended blueprint.

Lets have a look how the schema of `data/blueprints//discharge_summary.json` looks like. You can inspect multiple properties of the output below to get a base understanding of how a schema is defined.


JSON("data/blueprints/Check-blueprint.json")

In [7]:
project_arn = 'arn:aws:bedrock:us-east-1:578787088678:data-automation-project/fbb0cf14e381'

## Invoke Data Automation Async
With the data project configured, we can now invoke data automation for our sample document. When we submit the document for processing, BDA scans the file and splits it into individual documents based on contextand matches it against the list of blueprints provided.

In [8]:
response = bda_runtime_client.invoke_data_automation_async(
    inputConfiguration={
        's3Uri': document_s3_uri
    },
    outputConfiguration={
        's3Uri': bda_s3_output_location
    },
    dataAutomationConfiguration={
        'dataAutomationProjectArn': project_arn,
        'stage': 'LIVE'
    }, 
    dataAutomationProfileArn = f'arn:aws:bedrock:{current_region}:{account_id}:data-automation-profile/us.data-automation-v1'
)

invocationArn = response['invocationArn']
print(f'Invoked data automation job with invocation arn {invocationArn}')

Invoked data automation job with invocation arn arn:aws:bedrock:us-east-1:578787088678:data-automation-invocation/b5e92c60-4926-40ec-99f2-c47391d4bf62


### Get Data Automation Status

We can check the status and monitor the progress of the Invocation job using the `GetDataAutomationStatus`. This API takes the invocation arn we retrieved from the response to the `InvokeDataAutomationAsync` operation above.

The invocation job status moves from `Created` to `InProgress` and finally to `Success` when the job completes successfully, along with the S3 location of the results. If the job encounters and error the final status is either `ServiceError` or `ClientError` with error details

In [9]:
status_response = helper_functions.wait_for_completion(
            client=bda_client,
            get_status_function=bda_runtime_client.get_data_automation_status,
            status_kwargs={'invocationArn': invocationArn},
            completion_states=['Success'],
            error_states=['ClientError', 'ServiceError'],
            status_path_in_response='status',
            max_iterations=15,
            delay=30
)
if status_response['status'] == 'Success':
    job_metadata_s3_location = status_response['outputConfiguration']['s3Uri']
else:
    raise Exception(f'Invocation Job Error, error_type={status_response["error_type"]},error_message={status_response["error_message"]}')

Current status: InProgress. Waiting...
Operation completed successfully with status: Success


### View Job Metadata
Let's retrieve the job metadata. The Job metadata contains the S3 uri's for the standard output,custom output and the status of custom output. The custom output status could be either of `MATCH` or `NO_MATCH`. `MATCH` indicates BDA was able to find a matching blueprint for the specific segment from the list of blueprint we associated with the project. If BDA was unable to match the segment to a blueprint associated with the project then the `custom output status` is `NO_MATCH` and in this case BDA would only have a standard output extracted from that specific segment of the input file.

In [10]:
job_metadata = json.loads(helper_functions.read_s3_object(job_metadata_s3_location))

job_metadata_table = pd.DataFrame(job_metadata['output_metadata'][0]['segment_metadata']).fillna('')
job_metadata_table.index.name='Segment Index'
job_metadata_json = JSON(job_metadata, root="job_metadata", expanded=True)
# Display the widget
display_functions.display_multiple(
    [display_functions.get_view(job_metadata_table), display_functions.get_view(job_metadata_json)], 
    ["Table View", "Raw JSON"])

## Explore the Custom Insights

### View Segments and Matched Blueprints
As we can see in the `job metadata`, BDA creates a segment section each for each individual document that it has identified in the file. Each segment section has details on the matched blueprint and the results of the extraction. For each segment, BDA also outputs the page indices (one or more) from the original file.

We can now get the custom output corresponding to each segment and look at the insights that BDA custom output produces.

In [11]:
asset_id = 0
segments_metadata = next(item["segment_metadata"]
                                for item in job_metadata["output_metadata"] 
                                if item['asset_id'] == asset_id)

standard_outputs = [
    json.loads(helper_functions.read_s3_object(segment_metadata.get('standard_output_path')))for segment_metadata in segments_metadata]
custom_outputs = [json.loads(helper_functions.read_s3_object(segment_metadata.get('custom_output_path'))) if segment_metadata.get('custom_output_status') == 'MATCH' else None for segment_metadata in segments_metadata]


### View Custom output summary

In [12]:
custom_outputs_json = JSON(custom_outputs, root="custom_outputs", expanded=False)
custom_outputs_table = pd.DataFrame(helper_functions.get_summaries(custom_outputs)).fillna('')

display_functions.display_multiple(
    [
        display_functions.get_view(custom_outputs_table.style.hide(axis='index')),
        display_functions.get_view(custom_outputs_json)
    ], 
    ["Table View", "Raw JSON"])

### Explore Document Insights using Standard and Custom output

In [13]:
views=[]
titles=[]
# Use the function
for custom_output, standard_output in zip(custom_outputs, standard_outputs):
    if custom_output:
        result = helper_functions.transform_custom_output(custom_output['inference_result'], custom_output['explainability_info'][0])
        document_image_uris = [page.get('asset_metadata',{}).get('rectified_image') for page in standard_output.get('pages',[])]
        views += [display_functions.segment_view(document_image_uris=document_image_uris,
                   inference_result=result)]
        titles += [custom_output.get('matched_blueprint', {}).get('name', None)]
display_functions.display_multiple(views, titles)


Fields with confidence scores below 80% and require human verification:
Field: date, Confidence: 56.25%
Field: check_code, Confidence: 79.296875%
Field: payee_name, Confidence: 53.90625%
